In [1]:
import page

# -*- coding: utf-8 -*-
# page.encoding='utf-8'

from catmap import ReactionModel

mkm_file = 'cracking.mkm'
model = ReactionModel(setup_file=mkm_file)
model.output_variables += ['production_rate','rate','rate_control','coverage','selectivity_control','rxn_order']
model.run()

from catmap import analyze
vm = analyze.VectorMap(model)
vm.plot_variable = 'rate' #tell the model which output to plot
vm.log_scale = True #rates should be plotted on a log-scale
vm.min = 1e-25 #minimum rate to plot
vm.max = 1e3 #maximum rate to plot
vm.plot(save='rate.pdf') #draw the plot and save it as "rate.pdf"

vm.unique_only = False
vm.plot(save='all_rates.pdf')
vm.unique_only = True

vm.production_rate_map = model.production_rate_map #attach map
vm.threshold = 1e-30 #do not plot rates below this
vm.plot_variable = 'production_rate'
vm.plot(save='production_rate.pdf')

vm.descriptor_labels = ['H reactivity [eV]', 'C2H5 reactivity [eV]']
vm.subplots_adjust_kwargs = {'left':0.2,'right':0.8,'bottom':0.15}
vm.plot(save='pretty_production_rate.pdf')

vm.plot_variable = 'coverage'
vm.log_scale = False
vm.min = 0
vm.max = 1
vm.plot(save='coverage.pdf')

vm.include_labels = ['C2H5_s']
vm.plot(save='C2H5_coverage.pdf')

sa = analyze.ScalingAnalysis(model)
sa.plot(save='scaling.pdf')



Input line {'surface_name': 'None', 'site_name': 'gas', 'species_name': 'CH2OHCHOHCH2OH  -24.385', 'formation_energy': 'None', 'bulk_structure': '[]', 'frequencies': '[]', 'other_parameters': 'Input File Tutorial.'} does not have all required fields.  Ignoring.
Input line {'surface_name': 'None', 'site_name': 'gas', 'species_name': 'CH2OHCHOHCOOH  -22.689', 'formation_energy': 'None', 'bulk_structure': '[]', 'frequencies': '[]', 'other_parameters': 'Input File Tutorial.'} does not have all required fields.  Ignoring.
Input line {'surface_name': 'None', 'site_name': 'gas', 'species_name': 'H2O  -7.614', 'formation_energy': 'None', 'bulk_structure': '[]', 'frequencies': '[]', 'other_parameters': 'Input File Tutorial.'} does not have all required fields.  Ignoring.
Input line {'surface_name': 'None', 'site_name': 'gas', 'species_name': 'O2  -0.528', 'formation_energy': 'None', 'bulk_structure': '[]', 'frequencies': '[]', 'other_parameters': 'Input File Tutorial.'} does not have all requir

ValueError: No formation energy found for O2_g. Check input file.

In [ ]:
from ase.calculators.calculator import InputError
from glob import glob
import sys
from catmap.model import ReactionModel

model.output_variables += ['production_rate','rate','rate_control','coverage','selectivity_control','rxn_order']

output_variable = 'production_rate'
logfile = glob('*.log')
if len(logfile) > 1:
    raise InputError('Ambiguous logfile. Ensure that only one file ends with .log')
model = ReactionModel(setup_file=logfile[0])

if output_variable == 'rate_control':
    dim = 2
else:
    dim = 1

labels = model.output_labels['production_rate']

def flatten_2d(output):
    "Helper function for flattening rate_control output"
    flat = []
    for x in output:
        flat+= x
    return flat

#flatten rate_control labels
if output_variable == 'rate_control':
    flat_labels = []
    for i in labels[0]:
        for j in labels[1]:
            flat_labels.append('d'+i+'/d'+j)
    labels = flat_labels

#flatten elementary-step specific labels
if output_variable in ['rate','rate_constant','forward_rate_constant','reverse_rate_constant']:
    str_labels = []
    for label in labels:
        states = ['+'.join(s) for s in label]
        if len(states) == 2:
            new_label = '<->'.join(states)
        else:
            new_label = states[0]+'<->'+states[1]+'->'+states[2]
        str_labels.append(new_label)
    labels = str_labels

table = '\t'.join(list(['descriptor-'+d for d in model.descriptor_names])+list(labels))+'\n'

for pt, output in getattr(model,output_variable+'_map'):
    if dim == 2:
        output = flatten_2d(output)
    table += '\t'.join([str(float(i)) for i in pt+output])+'\n'

f = open(output_variable+'_table.txt','w')
f.write(table)
f.close()

In [ ]:
from catmap.model import ReactionModel

model = ReactionModel(setup_file='cracking.log')

#for MgO, cvgs in model.coverage_map:
   # print( 'descriptors:', MgO)
   #print( 'coverages', cvgs)
    
labels = model.output_labels['coverage']
for MgO ,cvg in model.coverage_map:
    print( 'descriptors',MgO)
    print( 'intermediates',labels)
    print( 'coverages', [float(c) for c in cvg])